# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [1]:
import pandas as pd
df = pd.read_csv('auto_dataset.csv')
df.head()

,brand,model,vehicleType,gearbox,fuelType,notRepairedDamage,powerPS,kilometer,autoAgeMonths,price
0,volkswagen,golf,kleinwagen,manuell,benzin,nein,75,150000,177,1500
1,skoda,fabia,kleinwagen,manuell,diesel,nein,69,90000,93,3600
2,bmw,3er,limousine,manuell,benzin,ja,102,150000,246,650
3,peugeot,2_reihe,cabrio,manuell,benzin,nein,109,150000,140,2200
4,mazda,3_reihe,limousine,manuell,benzin,nein,105,150000,136,2000


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [5]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

cat_cols = ['brand', 'model', 'vehicleType', 'gearbox', 'fuelType', 'notRepairedDamage']
num_cols = ['powerPS', 'kilometer', 'autoAgeMonths']

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(train[cat_cols])

cat_train = encoder.transform(train[cat_cols])
cat_val   = encoder.transform(val[cat_cols])
cat_test  = encoder.transform(test[cat_cols])

num_train = train[num_cols].values
num_val   = val[num_cols].values
num_test  = test[num_cols].values

mean = num_train.mean(axis=0)
std  = num_train.std(axis=0)

num_train_std = (num_train - mean) / std
num_val_std   = (num_val   - mean) / std
num_test_std  = (num_test  - mean) / std

X_train = np.hstack([cat_train, num_train_std])
X_val   = np.hstack([cat_val,   num_val_std])
X_test  = np.hstack([cat_test,  num_test_std])

y_train = train['price'].values
y_val   = val['price'].values
y_test  = test['price'].values

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

(800, 196) (100, 196) (100, 196)
(800,) (100,) (100,)


3. Разбейте датасет на train val test в отношении 8:1:1

In [4]:
train = df.iloc[:800]
val   = df.iloc[800:900]
test  = df.iloc[900:1000]

4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [6]:
import numpy as np
import pandas as pd

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

def gradient_mse(X, y, w):
    n = X.shape[0]
    return (2 / n) * X.T @ (X @ w - y)

def vgd_const(X, y, step, n_iter=2000):
    n, d = X.shape
    w = np.zeros(d)
    for k in range(n_iter):
        grad = gradient_mse(X, y, w)
        w = w - step * grad
    return w

grid = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]

rows = []
for n_step in grid:
    w = vgd_const(X_train, y_train, n_step)
    rows.append({
        'n': n_step,
        'loss_train': mse(y_train, X_train @ w),
        'r2_train':   r2(y_train, X_train @ w),
        'loss_val':   mse(y_val,   X_val   @ w),
    })

res_vgd_const = pd.DataFrame(rows)
print(res_vgd_const)

best_n_vgd = res_vgd_const.loc[res_vgd_const['loss_val'].idxmin(), 'n']
print("Лучший n:", best_n_vgd)

w_best = vgd_const(X_train, y_train, best_n_vgd, n_iter=2000)

loss_test = mse(y_test, X_test @ w_best)
r2_test   = r2(y_test, X_test @ w_best)

print(f"Loss_test: {loss_test:.2f}")
print(f"R2_test:   {r2_test:.4f}")

         n    loss_train  r2_train      loss_val
0  0.00001  9.507361e+07 -0.613949  1.058214e+08
1  0.00010  4.257705e+07  0.277221  4.759662e+07
2  0.00100  1.929006e+07  0.672536  2.394091e+07
3  0.01000  1.659144e+07  0.718347  2.418047e+07
4  0.10000  1.333218e+07  0.773676  2.597973e+07
5  1.00000           NaN       NaN           NaN
Лучший n: 0.001
Loss_test: 24186900.25
R2_test:   0.6752


5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
import numpy as np
import pandas as pd

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

def gradient_mse(X, y, w):
    n = X.shape[0]
    return (2 / n) * X.T @ (X @ w - y)

def eta_k(lam, k, s0=1.0, p=0.5):
    return lam * (s0 / (s0 + k)) ** p

def vgd_decay(X, y, lam, n_iter=2000, s0=1.0, p=0.5):
    n, d = X.shape
    w = np.zeros(d)
    for k in range(n_iter):
        grad = gradient_mse(X, y, w)
        w = w - eta_k(lam, k, s0, p) * grad
    return w

grid = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]

rows = []
for lam in grid:
    w = vgd_decay(X_train, y_train, lam)
    rows.append({
        'lambda': lam,
        'loss_train': mse(y_train, X_train @ w),
        'r2_train':   r2(y_train, X_train @ w),
        'loss_val':   mse(y_val,   X_val   @ w),
    })

res_vgd_decay = pd.DataFrame(rows)
print(res_vgd_decay)

best_lam_vgd = res_vgd_decay.loc[res_vgd_decay['loss_val'].idxmin(), 'lambda']
print("Лучшая λ:", best_lam_vgd)

w_best = vgd_decay(X_train, y_train, best_lam_vgd, n_iter=2000)

loss_test = mse(y_test, X_test @ w_best)
r2_test   = r2(y_test, X_test @ w_best)

print(f"Loss_test: {loss_test:.2f}")
print(f"R2_test:   {r2_test:.4f}")
print(f"Число итераций: 2000")

    lambda    loss_train  r2_train      loss_val
0  0.00001  1.061769e+08 -0.802437  1.182335e+08
1  0.00010  1.013728e+08 -0.720883  1.128610e+08
2  0.00100  6.669132e+07 -0.132137  7.420260e+07
3  0.01000  2.101489e+07  0.643256  2.480892e+07
4  0.10000  1.756727e+07  0.701782  2.405709e+07
5  1.00000  1.430182e+07  0.757215  2.494567e+07
Лучшая λ: 0.1
Loss_test: 21945633.77
R2_test:   0.7053
Число итераций: 2000


6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
import numpy as np
import pandas as pd

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

def gradient_mse_batch(X_batch, y_batch, w):
    n = X_batch.shape[0]
    return (2 / n) * X_batch.T @ (X_batch @ w - y_batch)

def sgd_const(X, y, step, batch_size=32, n_iter=2000, seed=42):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    w = np.zeros(d)
    for k in range(n_iter):
        idx = rng.choice(n, size=batch_size, replace=False)
        grad = gradient_mse_batch(X[idx], y[idx], w)
        w = w - step * grad
    return w

grid = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]

rows = []
for n_step in grid:
    w = sgd_const(X_train, y_train, n_step)
    rows.append({
        'n': n_step,
        'loss_train': mse(y_train, X_train @ w),
        'r2_train':   r2(y_train, X_train @ w),
        'loss_val':   mse(y_val,   X_val   @ w),
    })

res_sgd_const = pd.DataFrame(rows)
print(res_sgd_const)

best_n_sgd = res_sgd_const.loc[res_sgd_const['loss_val'].idxmin(), 'n']
print("Лучший n:", best_n_sgd)

w_best = sgd_const(X_train, y_train, best_n_sgd, n_iter=2000)

loss_test = mse(y_test, X_test @ w_best)
r2_test   = r2(y_test, X_test @ w_best)

print(f"Loss_test: {loss_test:.2f}")
print(f"R2_test:   {r2_test:.4f}")
print(f"Число итераций: 2000")

         n    loss_train  r2_train      loss_val
0  0.00001  9.517501e+07 -0.615670  1.059398e+08
1  0.00010  4.293008e+07  0.271228  4.796086e+07
2  0.00100  1.938524e+07  0.670920  2.386184e+07
3  0.01000  1.673266e+07  0.715950  2.451359e+07
4  0.10000  1.571426e+07  0.733238  3.070105e+07
5  1.00000           NaN       NaN           NaN
Лучший n: 0.001
Loss_test: 24236807.71
R2_test:   0.6746
Число итераций: 2000


7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
import numpy as np
import pandas as pd

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

def gradient_mse_batch(X_batch, y_batch, w):
    n = X_batch.shape[0]
    return (2 / n) * X_batch.T @ (X_batch @ w - y_batch)

def eta_k(lam, k, s0=1.0, p=0.5):
    return lam * (s0 / (s0 + k)) ** p

def sgd_decay(X, y, lam, batch_size=32, n_iter=2000, s0=1.0, p=0.5, seed=42):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    w = np.zeros(d)
    for k in range(n_iter):
        idx = rng.choice(n, size=batch_size, replace=False)
        grad = gradient_mse_batch(X[idx], y[idx], w)
        w = w - eta_k(lam, k, s0, p) * grad
    return w

grid = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]

rows = []
for lam in grid:
    w = sgd_decay(X_train, y_train, lam)
    rows.append({
        'lambda': lam,
        'loss_train': mse(y_train, X_train @ w),
        'r2_train':   r2(y_train, X_train @ w),
        'loss_val':   mse(y_val,   X_val   @ w),
    })

res_sgd_decay = pd.DataFrame(rows)
print(res_sgd_decay)

best_lam_sgd = res_sgd_decay.loc[res_sgd_decay['loss_val'].idxmin(), 'lambda']
print("Лучшая λ:", best_lam_sgd)

w_best = sgd_decay(X_train, y_train, best_lam_sgd, n_iter=2000)

loss_test = mse(y_test, X_test @ w_best)
r2_test   = r2(y_test, X_test @ w_best)

print(f"Loss_test: {loss_test:.2f}")
print(f"R2_test:   {r2_test:.4f}")
print(f"Число итераций: 2000")

    lambda    loss_train  r2_train      loss_val
0  0.00001  1.061838e+08 -0.802553  1.182426e+08
1  0.00010  1.014380e+08 -0.721989  1.129477e+08
2  0.00100  6.708207e+07 -0.138771  7.472092e+07
3  0.01000  2.109948e+07  0.641820  2.479724e+07
4  0.10000  1.757112e+07  0.701716  2.413173e+07
5  1.00000  1.479915e+07  0.748773  2.559508e+07
Лучшая λ: 0.1
Loss_test: 22099092.12
R2_test:   0.7033
Число итераций: 2000


8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
import numpy as np
import pandas as pd

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

def gradient_mse_one(x_i, y_i, w):
    return 2 * (x_i @ w - y_i) * x_i

def sag_const(X, y, step, n_iter=2000, seed=42):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    w = np.zeros(d)

    grad_table = np.zeros((n, d))
    for i in range(n):
        grad_table[i] = gradient_mse_one(X[i], y[i], w)

    g_mean = grad_table.mean(axis=0)

    for k in range(n_iter):
        j = rng.integers(0, n)
        g_new = gradient_mse_one(X[j], y[j], w)

        g_mean = g_mean + (g_new - grad_table[j]) / n
        grad_table[j] = g_new

        w = w - step * g_mean

    return w

grid = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]

rows = []
for n_step in grid:
    w = sag_const(X_train, y_train, n_step)
    rows.append({
        'n': n_step,
        'loss_train': mse(y_train, X_train @ w),
        'r2_train':   r2(y_train, X_train @ w),
        'loss_val':   mse(y_val,   X_val   @ w),
    })

res_sag_const = pd.DataFrame(rows)
print(res_sag_const)

best_n_sag = res_sag_const.loc[res_sag_const['loss_val'].idxmin(), 'n']
print("Лучший n:", best_n_sag)

w_best = sag_const(X_train, y_train, best_n_sag, n_iter=2000)

loss_test = mse(y_test, X_test @ w_best)
r2_test   = r2(y_test, X_test @ w_best)

print(f"Loss_test: {loss_test:.2f}")
print(f"R2_test:   {r2_test:.4f}")
print(f"Число итераций: 2000")

         n    loss_train      r2_train      loss_val
0  0.00001  9.485268e+07 -6.101985e-01  1.055767e+08
1  0.00010  3.647891e+07  3.807419e-01  4.109340e+07
2  0.00100  2.951890e+07  4.988936e-01  3.119675e+07
3  0.01000  1.538587e+08 -1.611871e+00  1.768184e+08
4  0.10000  1.168485e+11 -1.982594e+03  1.005115e+11
5  1.00000  2.191023e+22 -3.719433e+14  2.306930e+22
Лучший n: 0.001
Loss_test: 32287770.42
R2_test:   0.5665
Число итераций: 2000


9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [11]:
import numpy as np
import pandas as pd

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

def gradient_mse_one(x_i, y_i, w):
    return 2 * (x_i @ w - y_i) * x_i

def eta_k(lam, k, s0=1.0, p=0.5):
    return lam * (s0 / (s0 + k)) ** p

def sag_decay(X, y, lam, n_iter=2000, s0=1.0, p=0.5, seed=42):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    w = np.zeros(d)

    grad_table = np.zeros((n, d))
    for i in range(n):
        grad_table[i] = gradient_mse_one(X[i], y[i], w)

    g_mean = grad_table.mean(axis=0)

    for k in range(n_iter):
        j = rng.integers(0, n)
        g_new = gradient_mse_one(X[j], y[j], w)

        g_mean = g_mean + (g_new - grad_table[j]) / n
        grad_table[j] = g_new

        w = w - eta_k(lam, k, s0, p) * g_mean

    return w

grid = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]

rows = []
for lam in grid:
    w = sag_decay(X_train, y_train, lam)
    rows.append({
        'lambda': lam,
        'loss_train': mse(y_train, X_train @ w),
        'r2_train':   r2(y_train, X_train @ w),
        'loss_val':   mse(y_val,   X_val   @ w),
    })

res_sag_decay = pd.DataFrame(rows)
print(res_sag_decay)

best_lam_sag = res_sag_decay.loc[res_sag_decay['loss_val'].idxmin(), 'lambda']
print("Лучшая λ:", best_lam_sag)

w_best = sag_decay(X_train, y_train, best_lam_sag, n_iter=2000)

loss_test = mse(y_test, X_test @ w_best)
r2_test   = r2(y_test, X_test @ w_best)

print(f"Loss_test: {loss_test:.2f}")
print(f"R2_test:   {r2_test:.4f}")
print(f"Число итераций: 2000")

    lambda    loss_train   r2_train      loss_val
0  0.00001  1.061764e+08  -0.802428  1.182329e+08
1  0.00010  1.013249e+08  -0.720069  1.128078e+08
2  0.00100  6.387746e+07  -0.084370  7.111346e+07
3  0.01000  3.020101e+07   0.487314  3.599017e+07
4  0.10000  2.247220e+08  -2.814832  2.117884e+08
5  1.00000  2.711280e+09 -45.026096  2.364390e+09
Лучшая λ: 0.01
Loss_test: 31970327.54
R2_test:   0.5707
Число итераций: 2000


10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
import numpy as np
import pandas as pd

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

def gradient_mse(X, y, w):
    n = X.shape[0]
    return (2 / n) * X.T @ (X @ w - y)

def momentum_const(X, y, step, alpha=0.9, n_iter=2000):
    n, d = X.shape
    w = np.zeros(d)
    h = np.zeros(d)
    for k in range(n_iter):
        grad = gradient_mse(X, y, w)
        h = alpha * h + step * grad
        w = w - h
    return w

grid = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]

rows = []
for n_step in grid:
    w = momentum_const(X_train, y_train, n_step)
    rows.append({
        'n': n_step,
        'loss_train': mse(y_train, X_train @ w),
        'r2_train':   r2(y_train, X_train @ w),
        'loss_val':   mse(y_val,   X_val   @ w),
    })

res_mom_const = pd.DataFrame(rows)
print(res_mom_const)

best_n_mom = res_mom_const.loc[res_mom_const['loss_val'].idxmin(), 'n']
print("Лучший n:", best_n_mom)

w_best = momentum_const(X_train, y_train, best_n_mom, n_iter=2000)

loss_test = mse(y_test, X_test @ w_best)
r2_test   = r2(y_test, X_test @ w_best)

print(f"Loss_test: {loss_test:.2f}")
print(f"R2_test:   {r2_test:.4f}")
print(f"Число итераций: 2000")

         n    loss_train  r2_train      loss_val
0  0.00001  4.261391e+07  0.276596  4.763781e+07
1  0.00010  1.929198e+07  0.672504  2.394219e+07
2  0.00100  1.659563e+07  0.718276  2.418117e+07
3  0.01000  1.333268e+07  0.773667  2.598252e+07
4  0.10000  1.223091e+07  0.792371  2.890856e+07
5  1.00000           NaN       NaN           NaN
Лучший n: 0.0001
Loss_test: 24185683.41
R2_test:   0.6753
Число итераций: 2000


11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
import numpy as np
import pandas as pd

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

def gradient_mse(X, y, w):
    n = X.shape[0]
    return (2 / n) * X.T @ (X @ w - y)

def eta_k(lam, k, s0=1.0, p=0.5):
    return lam * (s0 / (s0 + k)) ** p

def momentum_decay(X, y, lam, alpha=0.9, n_iter=2000, s0=1.0, p=0.5):
    n, d = X.shape
    w = np.zeros(d)
    h = np.zeros(d)
    for k in range(n_iter):
        grad = gradient_mse(X, y, w)
        h = alpha * h + eta_k(lam, k, s0, p) * grad
        w = w - h
    return w

grid = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]

rows = []
for lam in grid:
    w = momentum_decay(X_train, y_train, lam)
    rows.append({
        'lambda': lam,
        'loss_train': mse(y_train, X_train @ w),
        'r2_train':   r2(y_train, X_train @ w),
        'loss_val':   mse(y_val,   X_val   @ w),
    })

res_mom_decay = pd.DataFrame(rows)
print(res_mom_decay)

best_lam_mom = res_mom_decay.loc[res_mom_decay['loss_val'].idxmin(), 'lambda']
print("Лучшая λ:", best_lam_mom)

w_best = momentum_decay(X_train, y_train, best_lam_mom, n_iter=2000)

loss_test = mse(y_test, X_test @ w_best)
r2_test   = r2(y_test, X_test @ w_best)

print(f"Loss_test: {loss_test:.2f}")
print(f"R2_test:   {r2_test:.4f}")
print(f"Число итераций: 2000")

    lambda    loss_train  r2_train      loss_val
0  0.00001  1.013833e+08 -0.721061  1.128728e+08
1  0.00010  6.668420e+07 -0.132017  7.419536e+07
2  0.00100  2.098506e+07  0.643762  2.477380e+07
3  0.01000  1.756654e+07  0.701794  2.406201e+07
4  0.10000  1.429754e+07  0.757288  2.495286e+07
5  1.00000  1.248796e+07  0.788007  2.789870e+07
Лучшая λ: 0.01
Loss_test: 21939725.71
R2_test:   0.7054
Число итераций: 2000


12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
import numpy as np
import pandas as pd

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

def gradient_mse(X, y, w):
    n = X.shape[0]
    return (2 / n) * X.T @ (X @ w - y)

def adam_const(X, y, step, beta1=0.9, beta2=0.999, eps=1e-8, n_iter=2000):
    n, d = X.shape
    w = np.zeros(d)
    m = np.zeros(d)
    v = np.zeros(d)
    for k in range(n_iter):
        grad = gradient_mse(X, y, w)
        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * grad ** 2
        m_hat = m / (1 - beta1 ** (k + 1))
        v_hat = v / (1 - beta2 ** (k + 1))
        w = w - step * m_hat / (np.sqrt(v_hat) + eps)
    return w

grid = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]

rows = []
for n_step in grid:
    w = adam_const(X_train, y_train, n_step)
    rows.append({
        'n': n_step,
        'loss_train': mse(y_train, X_train @ w),
        'r2_train':   r2(y_train, X_train @ w),
        'loss_val':   mse(y_val,   X_val   @ w),
    })

res_adam_const = pd.DataFrame(rows)
print(res_adam_const)

best_n_adam = res_adam_const.loc[res_adam_const['loss_val'].idxmin(), 'n']
print("Лучший n:", best_n_adam)

w_best = adam_const(X_train, y_train, best_n_adam, n_iter=2000)

loss_test = mse(y_test, X_test @ w_best)
r2_test   = r2(y_test, X_test @ w_best)

print(f"Loss_test: {loss_test:.2f}")
print(f"R2_test:   {r2_test:.4f}")
print(f"Число итераций: 2000")

         n    loss_train  r2_train      loss_val
0  0.00001  1.067275e+08 -0.811783  1.188494e+08
1  0.00010  1.067083e+08 -0.811457  1.188287e+08
2  0.00100  1.065161e+08 -0.808194  1.186215e+08
3  0.01000  1.046140e+08 -0.775905  1.165708e+08
4  0.10000  8.749083e+07 -0.485225  9.810546e+07
5  1.00000  2.494273e+07  0.576578  3.209808e+07
Лучший n: 1.0
Loss_test: 34754957.44
R2_test:   0.5333
Число итераций: 2000


13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [15]:
import numpy as np
import pandas as pd

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

def gradient_mse(X, y, w):
    n = X.shape[0]
    return (2 / n) * X.T @ (X @ w - y)

def eta_k(lam, k, s0=1.0, p=0.5):
    return lam * (s0 / (s0 + k)) ** p

def adam_decay(X, y, lam, beta1=0.9, beta2=0.999, eps=1e-8, n_iter=2000, s0=1.0, p=0.5):
    n, d = X.shape
    w = np.zeros(d)
    m = np.zeros(d)
    v = np.zeros(d)
    for k in range(n_iter):
        grad = gradient_mse(X, y, w)
        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * grad ** 2
        m_hat = m / (1 - beta1 ** (k + 1))
        v_hat = v / (1 - beta2 ** (k + 1))
        w = w - eta_k(lam, k, s0, p) * m_hat / (np.sqrt(v_hat) + eps)
    return w

grid = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]

rows = []
for lam in grid:
    w = adam_decay(X_train, y_train, lam)
    rows.append({
        'lambda': lam,
        'loss_train': mse(y_train, X_train @ w),
        'r2_train':   r2(y_train, X_train @ w),
        'loss_val':   mse(y_val,   X_val   @ w),
    })

res_adam_decay = pd.DataFrame(rows)
print(res_adam_decay)

best_lam_adam = res_adam_decay.loc[res_adam_decay['loss_val'].idxmin(), 'lambda']
print("Лучшая λ:", best_lam_adam)

w_best = adam_decay(X_train, y_train, best_lam_adam, n_iter=2000)

loss_test = mse(y_test, X_test @ w_best)
r2_test   = r2(y_test, X_test @ w_best)

print(f"Loss_test: {loss_test:.2f}")
print(f"R2_test:   {r2_test:.4f}")
print(f"Число итераций: 2000")

    lambda    loss_train  r2_train      loss_val
0  0.00001  1.067296e+08 -0.811818  1.188516e+08
1  0.00010  1.067287e+08 -0.811804  1.188507e+08
2  0.00100  1.067203e+08 -0.811660  1.188416e+08
3  0.01000  1.066357e+08 -0.810224  1.187504e+08
4  0.10000  1.057930e+08 -0.795920  1.178420e+08
5  1.00000  9.771378e+07 -0.658768  1.091294e+08
Лучшая λ: 1.0
Loss_test: 128262609.76
R2_test:   -0.7222
Число итераций: 2000


14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [19]:
import pandas as pd

comparison = pd.DataFrame([
    {
        'Метод': 'VanillaGradientDescent',
        'Лучший шаг': 'n = 0.001',
        'Loss_train': 19290060,
        'Loss_test':  24186900,
        'R²_train':   0.6725,
        'R²_test':    0.6752,
        'Итерации':   2000,
    },
    {
        'Метод': 'VanillaGradientDescent (decay)',
        'Лучший шаг': 'n(λ) = 0.1·(1/(1+k))^0.5',
        'Loss_train': 17567270,
        'Loss_test':  21945634,
        'R²_train':   0.7018,
        'R²_test':    0.7053,
        'Итерации':   2000,
    },
    {
        'Метод': 'StochasticGradientDescent',
        'Лучший шаг': 'n = 0.001',
        'Loss_train': 19385240,
        'Loss_test':  24236808,
        'R²_train':   0.6709,
        'R²_test':    0.6746,
        'Итерации':   2000,
    },
    {
        'Метод': 'StochasticGradientDescent (decay)',
        'Лучший шаг': 'n(λ) = 0.1·(1/(1+k))^0.5',
        'Loss_train': 17571120,
        'Loss_test':  22099092,
        'R²_train':   0.7017,
        'R²_test':    0.7033,
        'Итерации':   2000,
    },
    {
        'Метод': 'SAGDescent',
        'Лучший шаг': 'n = 0.001',
        'Loss_train': 29518900,
        'Loss_test':  32287770,
        'R²_train':   0.4989,
        'R²_test':    0.5665,
        'Итерации':   2000,
    },
    {
        'Метод': 'SAGDescent (decay)',
        'Лучший шаг': 'n(λ) = 0.01·(1/(1+k))^0.5',
        'Loss_train': 30201010,
        'Loss_test':  31970328,
        'R²_train':   0.4873,
        'R²_test':    0.5707,
        'Итерации':   2000,
    },
    {
        'Метод': 'MomentumDescent',
        'Лучший шаг': 'n = 0.0001',
        'Loss_train': 19291980,
        'Loss_test':  24185683,
        'R²_train':   0.6725,
        'R²_test':    0.6753,
        'Итерации':   2000,
    },
    {
        'Метод': 'MomentumDescent (decay)',
        'Лучший шаг': 'n(λ) = 0.01·(1/(1+k))^0.5',
        'Loss_train': 17566540,
        'Loss_test':  21939726,
        'R²_train':   0.7018,
        'R²_test':    0.7054,
        'Итерации':   2000,
    },
    {
        'Метод': 'Adam',
        'Лучший шаг': 'n = 1.0',
        'Loss_train': 24942730,
        'Loss_test':  34754957,
        'R²_train':   0.5766,
        'R²_test':    0.5333,
        'Итерации':   2000,
    },
    {
        'Метод': 'Adam (decay)',
        'Лучший шаг': 'n(λ) = 1.0·(1/(1+k))^0.5',
        'Loss_train': 97713780,
        'Loss_test':  128262610,
        'R²_train':   -0.6588,
        'R²_test':    -0.7222,
        'Итерации':   2000,
    },
])

pd.set_option('display.float_format', lambda x: f'{x:,.2f}' if abs(x) > 100 else f'{x:.2f}')
display(comparison)

,Метод,Лучший шаг,Loss_train,Loss_test,R²_train,R²_test,Итерации
0,VanillaGradientDescent,n = 0.001,19290060,24186900,0.67,0.68,2000
1,VanillaGradientDescent (decay),n(λ) = 0.1·(1/(1+k))^0.5,17567270,21945634,0.70,0.71,2000
2,StochasticGradientDescent,n = 0.001,19385240,24236808,0.67,0.67,2000
3,StochasticGradientDescent (decay),n(λ) = 0.1·(1/(1+k))^0.5,17571120,22099092,0.70,0.70,2000
4,SAGDescent,n = 0.001,29518900,32287770,0.50,0.57,2000
5,SAGDescent (decay),n(λ) = 0.01·(1/(1+k))^0.5,30201010,31970328,0.49,0.57,2000
6,MomentumDescent,n = 0.0001,19291980,24185683,0.67,0.68,2000
7,MomentumDescent (decay),n(λ) = 0.01·(1/(1+k))^0.5,17566540,21939726,0.70,0.71,2000
8,Adam,n = 1.0,24942730,34754957,0.58,0.53,2000
9,Adam (decay),n(λ) = 1.0·(1/(1+k))^0.5,97713780,128262610,-0.66,-0.72,2000


15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

In [ ]:
1. ну лучшим у нас получился метод MomentumDescent (decay) . ну вывод такой по столбцам loss_test (должен быть минимальным)  и у нас по таблице выше равен 17566540 и R^2_Test (наоборот чем больше тем лучше) как раз у нас по таблице равен 0.7018. так происходит потому что шаг у нас не только текущий градиент но и взвешенная сумма всех предыдущих градиентов что делает наш градиент более гладким и также если у нас градиент примерно не отклоняется (указывает в ожном направлении) то шаг растет. этот шаг лучший потому что если ставить меньше шаг то модель не успеет обучится. если ставить больше то модель переобучится. 
2. R^2 это простыми словами на сколько предсказание модели лучше чем среднее. если равно 1 то идеально. R<0- модель работует хужде чем просто среднее R^2 = 0 тоже самое что если бы мы всегда прдсказывали просто среднее. чем болиже к 1 тем лучше )
R^2 train это R^2 на обучающей выборке. Показывает, насколько модель подстроилась под данные на которых училась. R^2_test  R^2 на тестовой выборке показывает насколько модель работает на новых данных которых при обучении не было.
3 ну если смотреть только на R^2_train иодель может просто запомнить обучающие данные и R^2_train будет высокий а на новых данных  низкий что является прееробучением нащше модели 
если смотреть только на R^2_test то одно число может случайно оказаться высоким или низким из-за конкретной тестовой выборки и тогла не очень понятно модель действительно хороша или просто повезло

SyntaxError: invalid decimal literal (2623086775.py, line 1)